In [ ]:
pip install pandas streamlit

In [24]:
import ast
import pandas as pd

In [34]:
movies=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_movies.csv')
credits=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_credits.csv')


In [ ]:
movies = movies.merge(credits, left_on='original_title', right_on='title')
movies.drop(columns='original_title',inplace=True)


In [45]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')

In [48]:
movies = movies.rename(columns={'title_x': 'title'})

In [49]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [28]:
def convert_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:
        L.append(i['name'])
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [53]:
movies.isnull().sum()
movies.dropna(inplace=True)

In [54]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [29]:
def fetch_director(text):
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            return i['name']
    return ""

movies['crew'] = movies['crew'].apply(fetch_director)

In [33]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['crew'] = movies['crew'].apply(lambda x: [x])

AttributeError: 'float' object has no attribute 'split'

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

TypeError: can only concatenate str (not "list") to str

In [ ]:
import requests

API_KEY = "eb907f1e"

def fetch_poster(movie_title):
    url = f"http://www.omdbapi.com/?t={movie_title}&apikey={API_KEY}"
    
    try:
        data = requests.get(url, timeout=5).json()
        
        if data.get("Response") == "True":
            poster = data.get("Poster")
            
            # Handle cases where poster is "N/A"
            if poster and poster != "N/A":
                return poster
        
        return "https://via.placeholder.com/300x450?text=No+Image"
    
    except:
        return "https://via.placeholder.com/300x450?text=Error"

In [ ]:
print(fetch_poster("vikings"))

https://m.media-amazon.com/images/M/MV5BOTFmZmExYTEtYmE0Mi00MzRmLWE4ZDYtOThiNzNlOTIyODljXkEyXkFqcGc@._V1_SX300.jpg
